# 🕳️ General Relativistic Ray-Tracing (GRRT) & Numerical Relativity
### Interactive Google Colab Notebook for High-Performance Black Hole Physics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

This notebook allows you to simulate and render photorealistic **Black Hole General Relativistic Ray-Tracing (GRRT)** and **Numerical Relativity Initial Data** directly inside Google Colab without needing local hardware!

### 🔬 Key Physical & Optical Phenomena Simulated:
- **Curved Spacetime Ray Tracing**: RK4 integration of light geodesics in Schwarzschild & isotropic metrics.
- **Multi-Hit Photon Ring Resolution**: Resolves primary accretion disk images, secondary Einstein arches, and photon rings ($n=1, n=2$).
- **Relativistic Accretion Disk**: Novikov-Thorne thin disk model with Keplerian orbital shear ($\Omega \sim r^{-3/2}$).
- **Doppler Beaming & Gravitational Redshift**: Relativistic flux boost ($I_{\text{obs}} = g^4 I_{\text{emit}}$) causing asymmetric illumination on approaching/receding sides.
- **3+1 ADM Numerical Relativity Solver**: Puncture method solution for the Hamiltonian constraint equation of spatial metric perturbations.
- **Event Horizon Plunge**: Infalling observer geodesics crossing into interior spacetime ($r < 2M$).


---
## 🛠️ Step 1: Environment Setup & Hardware Diagnostics
Run this cell to install dependencies and verify CPU/GPU hardware acceleration.

In [ ]:
# Check available GPU hardware in Google Colab
!nvidia-smi

# Install core dependencies
!pip install -q numba numpy scipy matplotlib pillow imageio imageio-ffmpeg ipywidgets

---
## 📁 Step 2: Load Simulation Engine Files
Choose one of the 3 easy options below to load the simulation `.py` files into Colab:

In [ ]:
import os
import sys

# Check if project files exist in Colab. If missing, options to load:
if not os.path.exists('grrt_black_hole.py'):
    print('[!] grrt_black_hole.py not found in current directory.')
    print('👉 Option A: Upload your project .py files or ZIP to the left sidebar (Files 📁 tab).')
    print('👉 Option B: If hosted on GitHub, uncomment and run the line below:')
    # !git clone https://github.com/YOUR_GITHUB_USERNAME/black-hole-numerical-simulation.git .
else:
    print('[✓] All physics simulation modules loaded successfully!')

# Run quick diagnostic benchmark test
!python colab_runner.py --test

---
## 🎨 Step 3: Interactive Black Hole Studio (`ipywidgets`)
Use the interactive controls below to adjust camera parameters, inclination angle, and accretion disk settings, then view high-definition renders inline.

In [ ]:
import time
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import grrt_black_hole
import ipywidgets as widgets
from IPython.display import display

def render_colab_black_hole(width, height, inclination, distance, fov):
    print(f'[*] Rendering {width}x{height} black hole (inclination: {inclination} deg, distance: {distance}M)...')
    M = 1.0
    r_isco = 6.0 * M
    r_out = 26.0 * M
    t0 = time.time()
    hits, r1, p1, g1, r2, p2, g2 = grrt_black_hole.precompute_geodesics(
        width, height, fov, distance, inclination, M, r_isco, r_out, max_steps=2200
    )
    img = grrt_black_hole.shade_frame(hits, r1, p1, g1, r2, p2, g2, M, r_isco, r_out, time_phase=0.0)
    rgb = (np.clip(img, 0.0, 1.0) * 255).astype(np.uint8)
    t1 = time.time()
    
    plt.figure(figsize=(12, 7))
    plt.imshow(rgb)
    plt.title(f'GRRT Black Hole Simulation | {width}x{height} | Inclination: {inclination} deg | Render Time: {t1-t0:.2f}s')
    plt.axis('off')
    plt.show()

# Interactive UI Widgets
w_width = widgets.IntSlider(value=640, min=320, max=1280, step=80, description='Width:')
w_height = widgets.IntSlider(value=360, min=180, max=720, step=45, description='Height:')
w_incl = widgets.FloatSlider(value=85.0, min=5.0, max=89.0, step=5.0, description='Inclination:')
w_dist = widgets.FloatSlider(value=40.0, min=15.0, max=80.0, step=5.0, description='Distance (M):')
w_fov = widgets.FloatSlider(value=36.0, min=15.0, max=60.0, step=3.0, description='FOV:')

ui = widgets.VBox([w_width, w_height, w_incl, w_dist, w_fov])
out = widgets.interactive_output(render_colab_black_hole, {
    'width': w_width, 'height': w_height, 'inclination': w_incl, 'distance': w_dist, 'fov': w_fov
})
display(ui, out)

---
## 🌌 Step 4: Numerical Relativity -- 3+1 ADM Puncture Solver
Solve the non-linear elliptic Hamiltonian constraint equation $\bar{D}^2 u = -\beta (\alpha + \alpha u + 1)^{-7}$ for spatial hypersurface conformal factor deviation.

In [ ]:
import simulation
import plot_simulation
import importlib
importlib.reload(plot_simulation)

print('[*] Constructing Puncture Initial Data for a Black Hole with Linear Momentum (P_x = 1.0)...')
bh = simulation.Puncture(bh_location=(0, 0, 0), linear_momentum=(1.0, 0.0, 0.0), grid_dim=16, boundary=4.0)
bh.construct_solution(tol=1e-10, it_max=50)
bh.write_to_file()

data_file = 'simulation_data_16_4.0.data'
plot_simulation.puncture_plot(data_file, plot_file='colab_puncture_plot.png')

# Display generated 3D surface plot
img_puncture = Image.open('colab_puncture_plot.png')
plt.figure(figsize=(10, 8))
plt.imshow(img_puncture)
plt.title('Numerical Relativity: Conformal Deviation Surface u(x,y)')
plt.axis('off')
plt.show()

---
## 🪐 Step 5: Event Horizon Plunge & Infalling Observer View
Integrate light rays crossing the event horizon into interior Schwarzschild spacetime ($r < 2M$).

In [ ]:
import grrt_plunge

print('[*] Rendering Infalling Horizon Plunge View (r = 1.8M inside event horizon)...')
img_plunge, r_cam, is_inside = grrt_plunge.render_plunge_frame(
    width=640, height=360, fov_deg=45.0, t_sim=35.0, cam_start_r=40.0, M=1.0, r_isco=6.0, r_out=26.0, max_steps=1800
)
rgb_plunge = (np.clip(img_plunge, 0.0, 1.0) * 255).astype(np.uint8)

plt.figure(figsize=(12, 7))
plt.imshow(rgb_plunge)
plt.title(f'Horizon Crossing View | Observer Radius r = {r_cam:.2f}M | Inside Horizon: {is_inside}')
plt.axis('off')
plt.show()

---
## ⚡ Step 6: GPU Acceleration via Numba CUDA
If you are running on an NVIDIA GPU runtime in Google Colab (T4 / V100 / A100), run this cell to render photorealistic 1280x720 frames in seconds using CUDA hardware acceleration!

In [ ]:
# Run CUDA-accelerated GPU ray tracer
!python cuda_black_hole_ray_tracer.py

# Display CUDA GPU output image
from PIL import Image
import matplotlib.pyplot as plt
if os.path.exists('cuda_blackhole_render.png'):
    img_cuda = Image.open('cuda_blackhole_render.png')
    plt.figure(figsize=(14, 8))
    plt.imshow(img_cuda)
    plt.title('NVIDIA CUDA GPU Accelerated GRRT Black Hole (1280x720)')
    plt.axis('off')
    plt.show()

---
## 🎬 Step 7: Render Looping GIF / MP4 Video & Export to Google Drive
Generate a smooth rotating accretion disk GIF or MP4 video and save it to your Google Drive!

In [ ]:
# Render a 30-frame animated accretion disk GIF
print('[*] Rendering 30-frame animated accretion disk GIF...')
!python grrt_black_hole.py --gif --frames 30 --width 480 --height 270 --output black_hole_colab_animation.gif

# Display GIF in notebook
from IPython.display import Image as IPythonImage
display(IPythonImage(filename='black_hole_colab_animation.gif'))

In [ ]:
# Optional: Mount Google Drive to save high-res renders and videos permanently
# from google.colab import drive
# drive.mount('/content/drive')
# !cp black_hole_colab_animation.gif /content/drive/MyDrive/